# Agente **en vivo** + WSI vs sobre-rechazo en `microsoft/Aion-1.0-Instruct-2.3B` (Colab L4, OpenRouter)

Ciego de **receta**, pero con ojos: un modelo barato (DeepSeek V4.1 Flash por OpenRouter) trabaja **dentro de esta
máquina** con la GPU. Tiene una terminal y un Python persistente, el modelo Aion, el paquete `wsi/` y el enunciado
(`benchmarks/live_agent_refusal_aion/TASK.md`). Sin receta, sin AGENTS.md, sin internet. Puede cargar el modelo,
capturar, trazar, probar parches, ver qué falla y ajustar. Tiene 60 min. Luego el mismo juez de siempre evalúa su
`.moipatch` en prompts que nunca vio.

**Antes de correr:** en Colab, ícono de la llave (Secrets) → agrega `OPENROUTER_API_KEY` con tu key y activa el
acceso para este notebook. El agente **no** puede leer la key: sus comandos corren como otro usuario de Linux
(`agent`), sin esa variable y sin acceso a tu Drive, al repo ni a los prompts de prueba (se comprueba antes de empezar).

Costo esperado: ~$0.5–3 de OpenRouter. ~90–110 min en L4. Resultado:
`MyDrive/wsi_runs/FINALES/live_agent_refusal_aion.json` (+ registro completo del agente en la carpeta de la corrida).
El runtime se apaga solo al final.

In [ ]:
MODEL = "microsoft/Aion-1.0-Instruct-2.3B"
AGENT_MODEL = "deepseek/deepseek-v4.1-flash-1017"   # cualquier modelo de OpenRouter con herramientas
AGENT_MIN = 60
RUN_NAME = "live_agent_refusal_aion"
SHARED = "blind_agent_refusal_aion"   # si esa corrida ya hizo el censo, se reusa
REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work"
AG = "/content/agent"
HARNESS = "/content/harness"
J = f"{REPO}/scripts/eval_refusal_blind.py"
ARM = "live_" + AGENT_MODEL.split("/")[-1].replace(".", "_")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U "transformers>=5" accelerate safetensors pandas pyarrow huggingface_hub hf_transfer
import transformers, torch; print("transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime, time
from google.colab import drive, userdata
KEY = userdata.get("OPENROUTER_API_KEY")   # falla aquí si no está en Secrets
assert KEY, "falta OPENROUTER_API_KEY en Secrets"
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
for d in (FINALES, OUTD, W):
    os.makedirs(d, exist_ok=True)
for src in (f"{ROOT}/{SHARED}", OUTD):   # datos + censo ya hechos (misma semilla, mismos prompts)
    for f in glob.glob(f"{src}/*.json*"):
        if os.path.basename(f).startswith(("test_", "agent_", "gen_base_", "census", "eval_live", "live_")):
            shutil.copy2(f, W)
print(sorted(os.listdir(W)))

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
ENV = {**os.environ, "PYTHONPATH": f"{REPO}:{REPO}/scripts"}
ENV.pop("OPENROUTER_API_KEY", None)

def run_step(cmd, env=None, timeout=None):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env or ENV)
    t0 = time.time()
    for line in p.stdout:
        print(line, end="", flush=True)
        if timeout and time.time() - t0 > timeout:
            p.kill(); raise RuntimeError(f"timeout ({timeout} s)")
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

def sync():
    for f in glob.glob(f"{W}/*.json*") + glob.glob(f"{W}/*.moipatch") + glob.glob(f"{W}/*.md"):
        shutil.copy2(f, OUTD)

def save_final(res):
    res["_final"] = {"complete": True, "model": MODEL, "agent_model": AGENT_MODEL, "commit": COMMIT,
                     "notebook": "live_agent_refusal_aion_colab.ipynb",
                     "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                     "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
    dst = f"{FINALES}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w") as fh:
        json.dump(res, fh, indent=1, ensure_ascii=False)
    os.replace(dst + ".tmp", dst)
    print("\nGuardado final en Drive:", dst)

# datos + censo del modelo base (se salta lo que ya esté)
if not os.path.exists(f"{W}/test_overrefusal.jsonl"):
    run_step(["python", "-u", J, "prepare", "--work", W]); sync()
run_step(["python", "-u", J, "census", "--work", W, "--model", BASE]); sync()
CENSUS = json.load(open(f"{W}/census.json"))
print("censo:", CENSUS)
if not CENSUS["passed"]:
    save_final({"stopped_at": "census", "census": CENSUS})

In [ ]:
# caja de arena del agente: usuario aparte, sin key, sin Drive, sin repo, sin prompts de prueba
if CENSUS["passed"] and not os.path.exists(f"{W}/{ARM}.moipatch"):
    !id -u agent >/dev/null 2>&1 || useradd -m -s /bin/bash agent
    shutil.rmtree(AG, ignore_errors=True)
    os.makedirs(f"{AG}/data"); os.makedirs(f"{AG}/out")
    shutil.copytree(f"{REPO}/wsi", f"{AG}/wsi"); shutil.copytree(f"{REPO}/wsr", f"{AG}/wsr")
    shutil.copy2(f"{REPO}/benchmarks/live_agent_refusal_aion/TASK.md", AG)
    for f in glob.glob(f"{W}/agent_*.jsonl"):
        shutil.copy2(f, f"{AG}/data")
    os.makedirs(HARNESS, exist_ok=True)
    for f in ("live_agent.py", "live_agent_repl.py"):
        shutil.copy2(f"{REPO}/scripts/{f}", HARNESS)
    !chown -R agent:agent {AG} && chmod -R a+rX {BASE} {HARNESS} && chmod 700 {REPO} {W} /root
    # comprobaciones: el agente ve la GPU y el modelo, pero no Drive, ni el repo, ni los datos de prueba
    def as_agent(cmd):
        return subprocess.run(["runuser", "-u", "agent", "--", "bash", "-lc", cmd], capture_output=True, text=True)
    gpu = as_agent("python -c 'import torch; print(torch.cuda.is_available())'").stdout.strip()
    assert gpu == "True", f"el usuario agent no ve la GPU: {gpu}"
    assert as_agent(f"ls {BASE}/config.json").returncode == 0, "el usuario agent no puede leer el modelo"
    REMOUNT = False
    if as_agent("ls /content/drive/MyDrive").returncode == 0:   # Drive visible para otros usuarios: se desmonta
        drive.flush_and_unmount(); REMOUNT = True
    for p in ("/content/drive/MyDrive", REPO, W, "/root"):
        r = as_agent(f"ls {p}")
        assert r.returncode != 0, f"el usuario agent PUEDE leer {p}: no se corre"
    print("caja de arena OK: GPU sí, modelo sí, Drive/repo/prueba no", "| Drive desmontado" if REMOUNT else "")

In [ ]:
# el agente trabaja (60 min). Su registro va saliendo aquí.
if CENSUS["passed"] and not os.path.exists(f"{W}/{ARM}.moipatch"):
    env = {**ENV, "PYTHONPATH": AG, "OPENROUTER_API_KEY": KEY, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "HOME": "/home/agent"}
    try:
        run_step(["python", "-u", f"{HARNESS}/live_agent.py", "--model", AGENT_MODEL, "--workdir", AG,
                  "--task", f"{AG}/TASK.md", "--log", f"{W}/{ARM}_log.jsonl", "--summary", f"{W}/{ARM}_run.json",
                  "--time-limit-min", str(AGENT_MIN), "--run-as", "agent"], env=env, timeout=(AGENT_MIN + 20) * 60)
    finally:
        !pkill -u agent || true
        del env
        if globals().get("REMOUNT"):
            drive.mount("/content/drive")
        for src, dst in ((f"{AG}/out/patch.moipatch", f"{W}/{ARM}.moipatch"), (f"{AG}/out/notes.md", f"{W}/{ARM}_notes.md")):
            if os.path.exists(src):
                shutil.copy2(src, dst)
        sync()
print(json.load(open(f"{W}/{ARM}_run.json")) if os.path.exists(f"{W}/{ARM}_run.json") else "sin registro")

In [ ]:
# juez en los prompts de prueba
if CENSUS["passed"]:
    run_json = json.load(open(f"{W}/{ARM}_run.json")) if os.path.exists(f"{W}/{ARM}_run.json") else None
    if os.path.exists(f"{W}/{ARM}.moipatch"):
        run_step(["python", "-u", J, "eval", "--work", W, "--model", BASE, "--patch", f"{W}/{ARM}.moipatch", "--arm", ARM]); sync()
        ev = json.load(open(f"{W}/eval_{ARM}.json"))
        print(f"{ARM}: sobre-rechazo {ev['base_over_refusal']:.1%} -> {ev['over_refusal']:.1%} (recupera {ev['recovered']}) | "
              f"dañinos rechazados {ev['base_toxic_refusal']:.1%} -> {ev['toxic_refusal']:.1%} (mantiene {ev['toxic_kept']}) | "
              f"controles idénticos {ev['controls_identical']:.1%} | ÉXITO: {ev['success']}")
    else:
        ev = {"error": "the agent did not write out/patch.moipatch"}
    notes = open(f"{W}/{ARM}_notes.md").read() if os.path.exists(f"{W}/{ARM}_notes.md") else None
    save_final({"census": CENSUS, "arm": ARM, "agent_run": run_json, "eval": ev, "agent_notes": notes})

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")